# EA2 Unidad 1: Modelos de predicción y clasificación

Este notebook ejecuta el flujo completo de la actividad: carga automática desde Hugging Face, preprocesamiento, regresión lineal, clasificación, interpretación y generación de `resumen_modelos.txt`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.clases_po.datos.hf_loader import (
    DATASET_ID,
    DATASET_SPLIT,
    DATASET_URL,
    load_dataframe,
    prepare_data,
)
from src.clases_po.entrenamiento.regresion_lineal import train_and_evaluate as train_regression
from src.clases_po.entrenamiento.clasificacion_riesgo import train_and_evaluate as train_classification
from src.clases_po.reportes.reportes import build_summary, write_summary


/workspaces/ML_2026Q4_2/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Carga automática y construcción del DataFrame

Se utiliza el dataset `scikit-learn/student-alcohol-consumption` desde Hugging Face mediante código. El objetivo de regresión es `G3`, la calificación final continua. Para clasificación se construye `performance` a partir de la mediana de `G3`, usando `High` cuando `G3 >= mediana` y `Low` en caso contrario.


In [2]:
df_raw = load_dataframe()
print(f"Dataset: {DATASET_ID}")
print(f"Split: {DATASET_SPLIT}")
print(f"Dimensiones: {df_raw.shape}")
print(f"Columnas: {df_raw.columns.tolist()}")
print("Tipos de datos:")
print(df_raw.dtypes.to_string())
print("Muestra:")
display(df_raw.head())


Dataset: scikit-learn/student-alcohol-consumption
Split: train
Dimensiones: (1044, 33)
Columnas: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']
Tipos de datos:
school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc         

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


## 1. Preprocesamiento y división de datos

Se reemplazan los valores `?` por valores faltantes, se eliminan identificadores textuales y se eliminan duplicados. La imputación, codificación y escalado se ajustan únicamente con el conjunto de entrenamiento para evitar fuga de información. La columna `G3` no se usa como predictor porque se emplea para construir la etiqueta `performance`.


In [3]:
data = prepare_data(df_raw)
df = data["df"]
print(f"Duplicados antes de limpiar: {data['duplicates_before']}")
print("Nulos antes de imputar:")
print(data["missing_before"][data["missing_before"] > 0].to_string() or "Ninguno")
print(f"Dimensiones después de limpiar: {df.shape}")
print(f"Columnas después de limpiar: {df.columns.tolist()}")
print("Tipos de los predictores después de convertir:")
print(data["X"].dtypes.to_string())
print("Distribución de clases:")
print(f"Predictoras de clasificación (sin G3): {data['X'].columns.tolist()}")
print(data["y_cls"].value_counts().sort_index().to_string())
print(f"Regresión: train={len(data['y_train_r'])}, test={len(data['y_test_r'])}")
print(f"Clasificación: train={len(data['y_train_c'])}, test={len(data['y_test_c'])}")
print("Nulos después de imputar: 0")


Duplicados antes de limpiar: 0
Nulos antes de imputar:
Series([], )
Dimensiones después de limpiar: (1044, 34)
Columnas después de limpiar: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'performance']
Tipos de los predictores después de convertir:
school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic

## 2. Regresión lineal

Pregunta: ¿se puede predecir la calificación final `G3` a partir de las características del estudiante y sus hábitos? Se utiliza `LinearRegression` con sus parámetros predeterminados, una división 80/20 y semilla 42. MAE y RMSE miden el error en puntos de calificación; R² representa la proporción de variabilidad explicada.


In [4]:
regression = train_regression(data)
regression_model = regression["model"]
coefficients = regression["coefficients"]
mae = regression["mae"]
rmse = regression["rmse"]
r2 = regression["r2"]
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")
print("Coeficientes:")
print(coefficients.sort_values(ascending=False).to_string())
print(f"Intercepto: {regression_model.intercept_:.4f}")


MAE: 1.0480
RMSE: 1.7898
R²: 0.7928
Coeficientes:
G2                   3.092656
G1                   0.453351
guardian_other       0.336553
Fjob_at_home         0.239641
paid_no              0.185747
absences             0.143342
reason_course        0.131689
traveltime           0.112565
Fjob_health          0.109652
goout                0.106878
nursery_no           0.094827
Mjob_health          0.086190
Mjob_services        0.079312
activities_no        0.078910
famsup_yes           0.077015
address_U            0.075922
famrel               0.075715
Fjob_other           0.054518
sex_F                0.044843
reason_reputation    0.042258
Mjob_teacher         0.042133
Walc                 0.038391
romantic_no          0.036952
Dalc                 0.033818
school_MS            0.017895
internet_no          0.016995
Pstatus_A            0.014500
schoolsup_yes        0.009420
higher_no            0.008112
famsize_LE3          0.006387
famsize_GT3         -0.006387
Fedu                

## 3. Clasificación

Pregunta: ¿se puede clasificar un vehículo como de eficiencia `High` o `Low`? Se utiliza `RandomForestClassifier` con 100 árboles, profundidad máxima 5 y semilla 42. La matriz de confusión permite identificar aciertos y confusiones de cada clase.


In [5]:
classification = train_classification(data)
classification_model = classification["model"]
accuracy = classification["accuracy"]
precision = classification["precision"]
recall = classification["recall"]
f1 = classification["f1"]
confusion = classification["confusion"]
print(f"Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
display(classification["class_metrics"])
print("Matriz de confusión (filas reales, columnas predichas; [Low, High]):")
print(confusion)


Accuracy: 0.9234 | Precision: 0.9462 | Recall: 0.9318 | F1: 0.9389


,precision,recall,f1,support
Low,0.886076,0.909091,0.897436,77
High,0.946154,0.931818,0.938931,132


Matriz de confusión (filas reales, columnas predichas; [Low, High]):
[[ 70   7]
 [  9 123]]


## 4. Interpretación y resumen automático

Los coeficientes de la regresión se interpretan sobre predictores estandarizados. El resumen integra las decisiones de preprocesamiento, métricas, coeficientes, matriz de confusión, interpretaciones y limitaciones. Se escribe automáticamente en `resumen_modelos.txt`.


In [6]:
summary = build_summary(df_raw, data, regression, classification)
write_summary(summary, PROJECT_ROOT / "resumen_modelos.txt")
print(summary)
print("Archivo generado: resumen_modelos.txt")


RESUMEN AUTOMÁTICO DE MODELOS DE MACHINE LEARNING
Fecha de ejecución UTC: 2026-10-31T00:13:36.019811+00:00
Dataset: scikit-learn/student-alcohol-consumption
URL: https://huggingface.co/datasets/scikit-learn/student-alcohol-consumption
Split: train
Dimensiones iniciales: (1044, 33)
Dimensiones después de limpiar: (1044, 34)

PREPROCESAMIENTO
----------------
Nulos iniciales: {}
Duplicados iniciales: 0
Se limpiaron valores nulos y se eliminaron posibles duplicados o identificadores.
Las variables categóricas se codificaron con One-Hot Encoding y las numéricas se escalaron.
Imputadores, codificadores y escaladores se ajustaron únicamente con train.
La etiqueta performance usa la mediana de G3 (11.00): High >= mediana, Low < mediana.
Predictoras (codificadas): ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'school_GP', 'school_MS', 'sex_F', 'sex_M', 'address_R', 'address_U', 'famsize_GT3', 'fam